In [6]:
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
import json
from itertools import combinations
from scipy.stats import ttest_ind
import pandas as pd

Entire the desired output_directory, model_name, and metric we care about. Then run all the code below

In [7]:
output_dir = 'models'

output_dir = Path('/data/vision/polina/users/marcusbl/bin_class/outputs') / output_dir

model_name = 'model_loss'
metrics = ['auc', 'acc']

Parse results into Pandas DF

In [8]:
results = []

for test_dir in output_dir.iterdir():
    for run_dir in test_dir.iterdir():
        if not run_dir.is_dir():
            continue

        with open(run_dir / 'test_info' / 'metric_info.json') as f:
            metric_info = json.load(f)

        flat = {
            (model, metric): value
            for model, metrics in metric_info.items()
            for metric, value in metrics.items()
        }

        results.append({
            "test": test_dir.name,
            "run": run_dir.name,
            **flat
        })

df = pd.DataFrame(results).set_index(["test", "run"])

df.columns = pd.MultiIndex.from_tuples(df.columns)
df = df.sort_index(axis=1).sort_index()


In [9]:
df['model_auc']

acc    auc  epoch     f1  fn  fp    fpr   loss   prec  \
test     run                                                             
conv_nxt run0  0.859  0.931     -1  0.675  27  83  0.130  0.314  0.579   
         run1  0.887  0.946     -1  0.694  16  74  0.109  0.253  0.580   
         run2  0.915  0.959     -1  0.780  34  36  0.054  0.228  0.775   
         run3  0.883  0.941     -1  0.743  28  58  0.100  0.281  0.681   
         run4  0.922  0.934     -1  0.749  49  10  0.016  0.293  0.898   
resnet18 run0  0.897  0.906     -1  0.672  59  21  0.033  0.278  0.796   
         run1  0.932  0.942     -1  0.755  35  19  0.028  0.190  0.814   
         run2  0.907  0.938     -1  0.748  45  31  0.047  0.233  0.785   
         run3  0.843  0.910     -1  0.681  29  86  0.148  0.419  0.589   
         run4  0.918  0.944     -1  0.730  53   9  0.015  0.214  0.903   
resnet34 run0  0.901  0.913     -1  0.700  51  26  0.041  0.273  0.776   
         run1  0.926  0.948     -1  0.715  44  15  0.022  0.212  0.831   
         run2  0.918  0.949     -1  0.776  42  25  0.038  0.207  0.823   
         run3  0.831  0.921     -1  0.668  27  97  0.167  0.430  0.563   
         run4  0.918  0.934     -1  0.760  39  23  0.037  0.221  0.810   
resnet50 run0  0.888  0.918     -1  0.690  44  43  0.067  0.273  0.693   
         run1  0.956  0.966     -1  0.837  28   7  0.010  0.150  0.928   
         run2  0.926  0.957     -1  0.805  32  29  0.044  0.200  0.813   
         run3  0.868  0.939     -1  0.730  21  76  0.131  0.299  0.633   
         run4  0.919  0.951     -1  0.734  53   8  0.013  0.243  0.913   

               recall   tn   tp    tpr  
test     run                            
conv_nxt run0   0.809  556  114  0.809  
         run1   0.864  602  102  0.864  
         run2   0.785  626  124  0.785  
         run3   0.816  523  124  0.816  
         run4   0.642  610   88  0.642  
resnet18 run0   0.582  618   82  0.582  
         run1   0.703  657   83  0.703  
         run2   0.715  631  113  0.715  
         run3   0.809  495  123  0.809  
         run4   0.613  611   84  0.613  
resnet34 run0   0.638  613   90  0.638  
         run1   0.627  661   74  0.627  
         run2   0.734  637  116  0.734  
         run3   0.822  484  125  0.822  
         run4   0.715  597   98  0.715  
resnet50 run0   0.688  596   97  0.688  
         run1   0.763  669   90  0.763  
         run2   0.797  633  126  0.797  
         run3   0.862  505  131  0.862  
         run4   0.613  612   84  0.613

Average/Variance for desired metrics over all runs for the desired model

In [10]:
m = df[model_name]

stats = m.groupby(level='test').agg(['mean', 'var'])

stats = stats.sort_values(by=('auc', 'mean'), ascending=False)

stats[[(metric, stat) for metric in metrics for stat in ['mean', 'var']]]

auc               acc          
            mean       var    mean       var
test                                        
resnet50  0.9440  0.000313  0.9118  0.001076
conv_nxt  0.9342  0.000281  0.9126  0.000564
resnet34  0.9328  0.000340  0.9036  0.000426
resnet18  0.9264  0.000218  0.9046  0.000472

Paired T-test for signficance. Note that it's paired b/c same seed for each 

In [12]:
from itertools import combinations
from scipy.stats import ttest_rel

rows = []

tests = m.index.get_level_values("test").unique()

for metric in metrics:
    for test_a, test_b in combinations(tests, 2):
        a = m.loc[test_a, metric].dropna()
        b = m.loc[test_b, metric].dropna()

        # align by run name so only matching seeds are compared
        paired = pd.concat(
            [a.rename("a"), b.rename("b")],
            axis=1,
            join="inner"
        ).dropna()

        if len(paired) < 2:
            t_stat = np.nan
            p_value = np.nan
        else:
            t_stat, p_value = ttest_rel(paired["a"], paired["b"])

        rows.append({
            "metric": metric,
            "test_a": test_a,
            "test_b": test_b,
            "mean_a": paired["a"].mean() if len(paired) else np.nan,
            "mean_b": paired["b"].mean() if len(paired) else np.nan,
            "diff": (paired["a"] - paired["b"]).mean() if len(paired) else np.nan,
            "n_pairs": len(paired),
            "t_stat": t_stat,
            "p_value": p_value,
        })

paired_tstats = (
    pd.DataFrame(rows)
    .set_index(["metric", "test_a", "test_b"])
    .sort_values(["metric", "p_value"])
)

paired_tstats

mean_a  mean_b    diff  n_pairs    t_stat   p_value
metric test_a   test_b                                                       
acc    conv_nxt resnet18  0.9126  0.9046  0.0080        5  3.002347  0.039850
                resnet34  0.9126  0.9036  0.0090        5  1.888990  0.131904
       resnet18 resnet50  0.9046  0.9118 -0.0072        5 -1.271997  0.272294
       resnet34 resnet50  0.9036  0.9118 -0.0082        5 -1.012886  0.368404
       resnet18 resnet34  0.9046  0.9036  0.0010        5  0.294884  0.782754
       conv_nxt resnet50  0.9126  0.9118  0.0008        5  0.174824  0.869710
auc    resnet18 resnet50  0.9264  0.9440 -0.0176        5 -7.602042  0.001607
       resnet34 resnet50  0.9328  0.9440 -0.0112        5 -3.282778  0.030421
       conv_nxt resnet18  0.9342  0.9264  0.0078        5  2.326545  0.080556
       resnet18 resnet34  0.9264  0.9328 -0.0064        5 -1.835326  0.140357
       conv_nxt resnet50  0.9342  0.9440 -0.0098        5 -1.812330  0.144156
                resnet34  0.9342  0.9328  0.0014        5  0.229972  0.829395